### Level 3, Task 1: Predictive Modeling (Regression)

In [ ]:
%pip install scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# Ensure scikit-learn is available; provide install instructions if not
try:
    from sklearn.model_selection import train_test_split
    from sklearn.linear_model import LinearRegression
    from sklearn.metrics import mean_squared_error, r2_score
except ModuleNotFoundError:
    print("scikit-learn is not installed in this environment. Install it by running:")
    print("%pip install scikit-learn")
    raise

# 1. Load the dataset (file name contains a space before extension)
df = pd.read_csv('Dataset .csv')

# 2. Encode categorical features using the actual CSV column names
# CSV columns: 'Has Table booking', 'Has Online delivery', 'Price range', 'Aggregate rating'
df['table_booking_encoded'] = df['Has Table booking'].map({'Yes': 1, 'No': 0})
df['online_delivery_encoded'] = df['Has Online delivery'].map({'Yes': 1, 'No': 0})

# 3. Define Features (X) and Target (y) using CSV column names
X = df[['Price range', 'table_booking_encoded', 'online_delivery_encoded']]
y = df['Aggregate rating']

# 4. Split the data into 80% Training and 20% Testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Train the Linear Regression Model
model = LinearRegression()
model.fit(X_train, y_train)

# 6. Make predictions and evaluate performance
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Dataset loaded and model trained successfully!")
print(f"Training set size: {X_train.shape[0]} rows")
print(f"Testing set size: {X_test.shape[0]} rows\n")
print("--- Model Performance Evaluation ---")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"R-squared Score (R2):     {r2:.4f}")

### Level 3, Task 2: Customer Preference Analysis

In [ ]:
# 1. Analyze the relationship between cuisines and aggregate ratings
# Since a restaurant can have multiple cuisines separated by commas, we split them
cuisine_df = df.copy()
cuisine_df['Cuisines'] = cuisine_df['Cuisines'].fillna('Unknown')
cuisine_df['cuisine_list'] = cuisine_df['Cuisines'].apply(lambda x: [c.strip() for c in x.split(',')])

# Explode the list so each cuisine gets its own row for accurate calculations
exploded_cuisines = cuisine_df.explode('cuisine_list')

# 2. Find the top 10 most popular cuisines by restaurant count
top_cuisines = exploded_cuisines['cuisine_list'].value_counts().head(10)

print("--- Top 10 Most Popular Cuisines (By Count) ---")
print(top_cuisines)
print("\n")

# 3. Determine which cuisines have the highest average ratings
cuisine_ratings = exploded_cuisines.groupby('cuisine_list')['Aggregate rating'].agg(['mean', 'count'])
# Filter for cuisines that appear at least 20 times to avoid outliers
top_rated_cuisines = cuisine_ratings[cuisine_ratings['count'] >= 20].sort_values(by='mean', ascending=False).head(10)

print("--- Top 10 Highest Rated Cuisines (Min 20 Restaurants) ---")
for cuisine, row in top_rated_cuisines.iterrows():
    print(f"Cuisine: {cuisine:<15} | Avg Rating: {row['mean']:.2f} (Count: {int(row['count'])})")

### Level 3, Task 3: Data Visualization


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set up a clean plotting area for two side-by-side charts
plt.figure(figsize=(14, 6))

# 1. Histogram: Distribution of Aggregate Ratings
plt.subplot(1, 2, 1)
sns.histplot(data=df, x='Aggregate rating', bins=20, kde=True, color='skyblue')
plt.title('Distribution of Aggregate Ratings')
plt.xlabel('Aggregate Rating')
plt.ylabel('Frequency')

# 2. Bar Plot: Average Rating by Price Range
plt.subplot(1, 2, 2)
sns.barplot(data=df, x='Price range', y='Aggregate rating', palette='muted', errorbar=None)
plt.title('Average Rating by Price Range')
plt.xlabel('Price Range (1 = Cheapest, 4 = Most Expensive)')
plt.ylabel('Average Rating')

# Adjust layout and save the charts to a file
plt.tight_layout()
plt.savefig('rating_analysis.png', dpi=300, bbox_inches='tight')
print('Saved plot to rating_analysis.png')
plt.show()